# PyTorch Training Pipeline — Breast Cancer Classification

**A reproducible, end-to-end binary classification workflow with interactive Plotly visualizations.**

This notebook upgrades a simple sigmoid-based PyTorch classifier into a structured machine-learning workflow: data checks, exploratory analysis, leakage-aware preprocessing, reproducible train/validation/test splits, model comparison, early stopping, threshold analysis, evaluation, and artifact export.

> **Important educational-use notice:** This is a learning project built from a public benchmark dataset. Its outputs are not medical advice, are not validated for clinical use, and must not be used to diagnose or make decisions about any person.

| Project detail | Value |
|---|---|
| Task | Binary classification |
| Positive class | Malignant (`M`, encoded as `1`) |
| Negative class | Benign (`B`, encoded as `0`) |
| Models | PyTorch logistic-regression baseline and multilayer perceptron (MLP) |
| Visualizations | Interactive Plotly charts, saved as HTML |
| Main goal | Learn the complete training and evaluation workflow, not build a clinical tool |

---

## Notebook roadmap

1. Environment, imports, and reproducibility
2. Load and validate the dataset
3. Interactive exploratory data analysis (EDA)
4. Split data and fit preprocessing on training data only
5. Build PyTorch tensors and data loaders
6. Train two models with early stopping
7. Compare models on validation data and select a threshold
8. Evaluate once on the held-out test set
9. Explain model behavior with permutation importance
10. Run a sample prediction and save reusable artifacts

## 1. Environment and imports

The notebook uses standard scientific Python libraries plus PyTorch and Plotly. In a fresh environment, install missing packages in a separate cell with `%pip install torch scikit-learn pandas numpy plotly joblib`.

In [1]:
from __future__ import annotations

import copy
import json
import random
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print(f"PyTorch: {torch.__version__}")
print(f"pandas:  {pd.__version__}")
print(f"NumPy:   {np.__version__}")
print(f"Plotly:  {px.__version__ if hasattr(px, '__version__') else 'available'}")

PyTorch: 2.11.0+cu130
pandas:  2.2.3
NumPy:   2.1.3
Plotly:  available


## 2. Reproducibility and project settings

A fixed random seed helps make the split, initialization, and shuffled mini-batches repeatable. Results can still vary slightly across hardware and library versions.

In [2]:
SEED = 42
TEST_SIZE = 0.15
VALIDATION_SIZE_WITHIN_DEVELOPMENT = 0.1765  # approximately 15% of the full dataset
BATCH_SIZE = 32
MAX_EPOCHS = 120
EARLY_STOPPING_PATIENCE = 18
LEARNING_RATE = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("outputs")
PLOT_DIR = OUTPUT_DIR / "plots"
ARTIFACT_DIR = Path("artifacts")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PLOTLY_CONFIG = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": True,
    "modeBarButtonsToRemove": ["lasso2d", "select2d"],
}


def set_seed(seed: int = SEED) -> None:
    """Set common random seeds for repeatable experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def publish_plot(fig: go.Figure, filename: str) -> Path:
    """Apply consistent styling, show an interactive chart, and save it as HTML."""
    fig.update_layout(
        template="plotly_white",
        font={"family": "Arial, sans-serif", "size": 12},
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin={"l": 60, "r": 35, "t": 75, "b": 60},
        hoverlabel={"font_size": 12},
        legend={"title": "", "bgcolor": "rgba(255,255,255,0.75)"},
    )
    path = PLOT_DIR / f"{filename}.html"
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    fig.show(config=PLOTLY_CONFIG)
    print(f"Interactive chart saved: {path}")
    return path


set_seed()
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
print(f"Compute device: {DEVICE}")
print(f"Interactive Plotly charts will be saved under: {PLOT_DIR.resolve()}")

Compute device: cuda
Interactive Plotly charts will be saved under: /content/outputs/plots


## 3. Load the dataset

The first attempt uses the same public CSV source as the original notebook. If that source is unavailable, the code falls back to scikit-learn's bundled Wisconsin breast-cancer dataset and normalizes its labels to the same convention (`M` = malignant, `B` = benign).

The notebook removes identifier/empty columns, validates the label values, and keeps the numeric measurements as candidate features.

In [3]:
DATA_URL = "https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv"
DATA_SOURCE = DATA_URL

try:
    import requests
    response = requests.get(DATA_URL, timeout=15)
    response.raise_for_status()
    from io import StringIO
    raw_df = pd.read_csv(StringIO(response.text))
    print("Loaded dataset from the original public CSV source.")
except Exception as exc:
    print(f"Remote CSV could not be loaded ({type(exc).__name__}: {exc}).")
    print("Using scikit-learn's built-in copy of the Wisconsin breast-cancer dataset instead.")
    bundled = load_breast_cancer(as_frame=True)
    raw_df = bundled.data.copy()
    # In scikit-learn's dataset, target 0 is malignant and target 1 is benign.
    raw_df.insert(0, "diagnosis", np.where(bundled.target.to_numpy() == 0, "M", "B"))
    DATA_SOURCE = "scikit-learn: load_breast_cancer (bundled dataset)"

raw_df.columns = [str(column).strip() for column in raw_df.columns]
target_candidates = [column for column in raw_df.columns if column.lower() == "diagnosis"]
if not target_candidates:
    raise ValueError("A 'diagnosis' target column was not found in the downloaded or fallback dataset.")
TARGET_COLUMN = target_candidates[0]

# Remove identifiers and all-empty placeholder columns (e.g. an unnamed CSV index column).
columns_to_drop = [
    column for column in raw_df.columns
    if column.lower() in {"id", "unnamed: 32", "unnamed: 0"} or column.lower().startswith("unnamed:")
]
df = raw_df.drop(columns=columns_to_drop, errors="ignore").copy()
raw_labels = df[TARGET_COLUMN].astype(str).str.strip().str.upper()
valid_label_mask = raw_labels.isin(["B", "M"])
df = df.loc[valid_label_mask].copy()
raw_labels = raw_labels.loc[valid_label_mask]
df["target"] = (raw_labels == "M").astype("int64")
df["class_label"] = df["target"].map({0: "Benign", 1: "Malignant"})

FEATURE_COLUMNS = [column for column in df.columns if column not in {TARGET_COLUMN, "target", "class_label"}]
X = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = df["target"].astype("int64")

if X.shape[1] == 0:
    raise ValueError("No numeric feature columns were found.")
if y.nunique() != 2:
    raise ValueError(f"Expected two classes, but found {sorted(y.unique().tolist())}.")

class_counts = y.value_counts().sort_index()
print(f"Dataset source: {DATA_SOURCE}")
print(f"Rows: {len(df):,} | Candidate features: {len(FEATURE_COLUMNS)}")
print("Target mapping: 0 = Benign (B), 1 = Malignant (M)")

Loaded dataset from the original public CSV source.
Dataset source: https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv
Rows: 569 | Candidate features: 30
Target mapping: 0 = Benign (B), 1 = Malignant (M)


In [4]:
print("Dataset preview")
display(df.head().style.set_caption("First five observations").format(precision=3))

profile = pd.DataFrame({
    "Data type": X.dtypes.astype(str),
    "Missing values": X.isna().sum(),
    "Unique values": X.nunique(dropna=True),
})
display(profile.head(12))
print(f"Full shape after identifier cleanup: {df.shape}")
print(f"Exact duplicate rows: {int(df.duplicated().sum())}")

Dataset preview


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,target,class_label
0,M,17.990,10.380,122.800,1001.000,0.118,0.278,0.300,0.147,0.242,0.079,1.095,0.905,8.589,153.400,0.006,0.049,0.054,0.016,0.030,0.006,25.380,17.330,184.600,2019.000,0.162,0.666,0.712,0.265,0.460,0.119,1,Malignant
1,M,20.570,17.770,132.900,1326.000,0.085,0.079,0.087,0.070,0.181,0.057,0.543,0.734,3.398,74.080,0.005,0.013,0.019,0.013,0.014,0.004,24.990,23.410,158.800,1956.000,0.124,0.187,0.242,0.186,0.275,0.089,1,Malignant
2,M,19.690,21.250,130.000,1203.000,0.110,0.160,0.197,0.128,0.207,0.060,0.746,0.787,4.585,94.030,0.006,0.040,0.038,0.021,0.022,0.005,23.570,25.530,152.500,1709.000,0.144,0.424,0.450,0.243,0.361,0.088,1,Malignant
3,M,11.420,20.380,77.580,386.100,0.142,0.284,0.241,0.105,0.260,0.097,0.496,1.156,3.445,27.230,0.009,0.075,0.057,0.019,0.060,0.009,14.910,26.500,98.870,567.700,0.210,0.866,0.687,0.258,0.664,0.173,1,Malignant
4,M,20.290,14.340,135.100,1297.000,0.100,0.133,0.198,0.104,0.181,0.059,0.757,0.781,5.438,94.440,0.011,0.025,0.057,0.019,0.018,0.005,22.540,16.670,152.200,1575.000,0.137,0.205,0.400,0.163,0.236,0.077,1,Malignant


,Data type,Missing values,Unique values
radius_mean,float64,0,456
texture_mean,float64,0,479
perimeter_mean,float64,0,522
area_mean,float64,0,539
smoothness_mean,float64,0,474
compactness_mean,float64,0,537
concavity_mean,float64,0,537
concave points_mean,float64,0,542
symmetry_mean,float64,0,432
fractal_dimension_mean,float64,0,499


Full shape after identifier cleanup: (569, 33)
Exact duplicate rows: 0


## 4. Exploratory data analysis (EDA)

EDA is used here to understand the data—not to fit transformations. The imputer and scaler will be fitted later on the training split only to prevent information leakage into validation/test data.

In [5]:
# Interactive class balance chart
class_summary = (
    df["class_label"].value_counts()
    .reindex(["Benign", "Malignant"])
    .rename_axis("Class")
    .reset_index(name="Count")
)
class_summary["Percentage"] = 100 * class_summary["Count"] / class_summary["Count"].sum()
fig = px.bar(
    class_summary,
    x="Class",
    y="Count",
    text=class_summary.apply(lambda row: f"{int(row['Count'])} · {row['Percentage']:.1f}%", axis=1),
    color="Class",
    title="Target Distribution — Benign vs Malignant",
    labels={"Count": "Number of observations"},
)
fig.update_traces(textposition="outside", marker_line_width=0)
fig.update_yaxes(rangemode="tozero")
fig.update_layout(showlegend=False, title_x=0.02)
publish_plot(fig, "01_target_distribution")
display(class_summary)

Interactive chart saved: outputs/plots/01_target_distribution.html


,Class,Count,Percentage
0,Benign,357,62.741652
1,Malignant,212,37.258348


In [6]:
# Missing values: if the dataset is complete, communicate that clearly instead of rendering a misleading empty chart.
missing_counts = X.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if len(missing_counts):
    fig = px.bar(
        missing_counts.head(20).rename_axis("Feature").reset_index(name="Missing values"),
        x="Missing values", y="Feature", orientation="h",
        title="Features with Missing Values (Top 20)",
        color="Missing values", color_continuous_scale="Blues",
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)
else:
    fig = go.Figure()
    fig.add_annotation(
        text="No missing feature values detected in the raw numeric columns.",
        x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
        font={"size": 18},
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.update_layout(title="Data Quality Check — Missing Values")
publish_plot(fig, "02_missing_values")

Interactive chart saved: outputs/plots/02_missing_values.html


PosixPath('outputs/plots/02_missing_values.html')

In [7]:
# Compare selected, interpretable measurements. Facets keep each feature's own numeric scale.
preferred_features = ["radius_mean", "texture_mean", "perimeter_mean", "area_mean"]
selected_features = [feature for feature in preferred_features if feature in X.columns]
if len(selected_features) < 2:
    selected_features = X.columns[:min(4, len(X.columns))].tolist()

plot_df = df[selected_features + ["class_label"]].melt(
    id_vars="class_label", var_name="Feature", value_name="Value"
).dropna(subset=["Value"])
fig = px.histogram(
    plot_df,
    x="Value",
    color="class_label",
    facet_col="Feature",
    facet_col_wrap=2,
    marginal="box",
    barmode="overlay",
    opacity=0.62,
    histnorm="probability density",
    title="Selected Feature Distributions by Diagnosis",
    labels={"class_label": "Diagnosis", "Value": "Measured value"},
)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.replace("Feature=", "")))
fig.update_layout(height=720, title_x=0.02, legend_title_text="Diagnosis")
publish_plot(fig, "03_feature_distributions")

Interactive chart saved: outputs/plots/03_feature_distributions.html


PosixPath('outputs/plots/03_feature_distributions.html')

In [8]:
# Display the 12 measurements most correlated with the binary target for a compact, readable heatmap.
eda_corr_frame = X.copy()
eda_corr_frame["target_malignant"] = y.to_numpy()
correlation_with_target = eda_corr_frame.corr(numeric_only=True)["target_malignant"].drop("target_malignant")
heatmap_features = correlation_with_target.abs().sort_values(ascending=False).head(12).index.tolist()
heatmap_matrix = eda_corr_frame[heatmap_features + ["target_malignant"]].corr(numeric_only=True)
fig = go.Figure(data=go.Heatmap(
    z=heatmap_matrix.to_numpy(),
    x=heatmap_matrix.columns.tolist(),
    y=heatmap_matrix.index.tolist(),
    zmin=-1, zmax=1, zmid=0,
    colorscale="RdBu_r",
    colorbar={"title": "Pearson r"},
    hovertemplate="%{y} × %{x}<br>Correlation: %{z:.3f}<extra></extra>",
))
fig.update_layout(
    title="Correlation Heatmap — Most Target-Associated Features",
    height=650,
    xaxis={"tickangle": -40},
    title_x=0.02,
)
publish_plot(fig, "04_correlation_heatmap")

Interactive chart saved: outputs/plots/04_correlation_heatmap.html


PosixPath('outputs/plots/04_correlation_heatmap.html')

## 5. Train / validation / test split

The test set is held back until the final evaluation. Model selection and threshold selection use the validation set. This separation gives a more honest estimate than repeatedly tuning against the test set.

- **Training set:** learns model parameters and preprocessing statistics.
- **Validation set:** supports early stopping and model/threshold selection.
- **Test set:** used at the end for the final, held-out report.

In [9]:
# First hold out the test set, then carve a validation set from the remaining development data.
X_development_raw, X_test_raw, y_development, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_development_raw, y_development,
    test_size=VALIDATION_SIZE_WITHIN_DEVELOPMENT,
    random_state=SEED,
    stratify=y_development,
)

def split_summary(name: str, labels: pd.Series) -> dict:
    return {
        "Split": name,
        "Rows": len(labels),
        "Benign": int((labels == 0).sum()),
        "Malignant": int((labels == 1).sum()),
        "Malignant %": round(100 * float(labels.mean()), 2),
    }

split_df = pd.DataFrame([
    split_summary("Train", y_train),
    split_summary("Validation", y_val),
    split_summary("Test (held out)", y_test),
])
display(split_df)

fig = px.bar(
    split_df.melt(id_vars="Split", value_vars=["Benign", "Malignant"], var_name="Class", value_name="Count"),
    x="Split", y="Count", color="Class", barmode="group",
    title="Class Counts Across the Three Data Splits",
    text_auto=True,
)
fig.update_layout(title_x=0.02)
publish_plot(fig, "05_split_class_counts")

,Split,Rows,Benign,Malignant,Malignant %
0,Train,397,249,148,37.28
1,Validation,86,54,32,37.21
2,Test (held out),86,54,32,37.21


Interactive chart saved: outputs/plots/05_split_class_counts.html


PosixPath('outputs/plots/05_split_class_counts.html')

## 6. Preprocessing without leakage

The median imputer and standard scaler are **fitted only on training data**. Validation and test rows are transformed with those already-fitted objects. Standardization is useful for neural networks because input features can have very different numeric scales.

In [10]:
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_train_raw)
X_val_imputed = imputer.transform(X_val_raw)
X_test_imputed = imputer.transform(X_test_raw)

X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)
X_val_scaled = scaler.transform(X_val_imputed).astype(np.float32)
X_test_scaled = scaler.transform(X_test_imputed).astype(np.float32)

y_train_np = y_train.to_numpy(dtype=np.float32).reshape(-1, 1)
y_val_np = y_val.to_numpy(dtype=np.int64)
y_test_np = y_test.to_numpy(dtype=np.int64)

print("Scaled array shapes")
print(f"X_train: {X_train_scaled.shape} | y_train: {y_train_np.shape}")
print(f"X_val:   {X_val_scaled.shape} | y_val:   {y_val_np.shape}")
print(f"X_test:  {X_test_scaled.shape} | y_test:  {y_test_np.shape}")
print(f"NaNs remaining after median imputation: {int(np.isnan(X_train_scaled).sum() + np.isnan(X_val_scaled).sum() + np.isnan(X_test_scaled).sum())}")

Scaled array shapes
X_train: (397, 30) | y_train: (397, 1)
X_val:   (86, 30) | y_val:   (86,)
X_test:  (86, 30) | y_test:  (86,)
NaNs remaining after median imputation: 0


In [11]:
# PCA is fitted on the training split only, then used to project training rows into two dimensions for visualization.
pca = PCA(n_components=2, random_state=SEED)
train_pca = pca.fit_transform(X_train_scaled)
pca_df = pd.DataFrame({
    "PC1": train_pca[:, 0],
    "PC2": train_pca[:, 1],
    "Diagnosis": y_train.map({0: "Benign", 1: "Malignant"}).to_numpy(),
})
explained = pca.explained_variance_ratio_ * 100
fig = px.scatter(
    pca_df, x="PC1", y="PC2", color="Diagnosis",
    title="PCA View of Training Data (2D Projection)",
    labels={"PC1": f"PC1 ({explained[0]:.1f}% variance)", "PC2": f"PC2 ({explained[1]:.1f}% variance)"},
    opacity=0.78,
    hover_data={"PC1": ":.2f", "PC2": ":.2f"},
)
fig.update_traces(marker={"size": 9, "line": {"width": 0.5, "color": "white"}})
fig.update_layout(title_x=0.02)
publish_plot(fig, "06_pca_training_projection")
print(f"Total variance explained by the first two PCs: {explained.sum():.2f}%")

Interactive chart saved: outputs/plots/06_pca_training_projection.html
Total variance explained by the first two PCs: 62.70%


## 7. Convert arrays to PyTorch tensors

The network returns **logits** (raw scores). `BCEWithLogitsLoss` combines the sigmoid operation and binary cross-entropy in a numerically stable way, so the model does not manually apply sigmoid before calculating the loss.

In [12]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_np.astype(np.float32).reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_np.astype(np.float32).reshape(-1, 1), dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)

print(f"Input tensor: {tuple(X_train_tensor.shape)}")
print(f"Target tensor: {tuple(y_train_tensor.shape)}")
print(f"Mini-batches per training epoch: {len(train_loader)}")

Input tensor: (397, 30)
Target tensor: (397, 1)
Mini-batches per training epoch: 13


## 8. Model architectures

Two models are trained on the same splits and scaled features:

1. **PyTorch logistic-regression baseline:** one linear layer. It is a useful benchmark and closely reflects the original notebook's single weight matrix and bias.
2. **Multilayer perceptron (MLP):** two hidden layers with ReLU activation and dropout, allowing the model to learn nonlinear interactions.

Both return one logit per observation. The architecture comparison is educational; the model with the best validation ROC-AUC is selected, rather than assuming the deeper model must be better.

In [13]:
class TorchLogisticRegression(nn.Module):
    """Single linear layer for binary classification; outputs logits."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


class TorchMLP(nn.Module):
    """Small multilayer perceptron for tabular binary classification."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(p=0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=0.10),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def build_model(model_name: str, input_dim: int) -> nn.Module:
    """Construct a fresh instance of the requested model."""
    if model_name == "Logistic Regression":
        return TorchLogisticRegression(input_dim)
    if model_name == "MLP":
        return TorchMLP(input_dim)
    raise ValueError(f"Unknown model name: {model_name}")


def predict_probabilities(model: nn.Module, features: torch.Tensor) -> np.ndarray:
    """Return positive-class probabilities in row order."""
    model.eval()
    with torch.no_grad():
        logits = model(features.to(DEVICE)).reshape(-1)
        probabilities = torch.sigmoid(logits).detach().cpu().numpy()
    return probabilities


print("Model input features:", len(FEATURE_COLUMNS))
print("Logistic model parameter count:", sum(p.numel() for p in TorchLogisticRegression(len(FEATURE_COLUMNS)).parameters()))
print("MLP parameter count:", sum(p.numel() for p in TorchMLP(len(FEATURE_COLUMNS)).parameters()))

Model input features: 30
Logistic model parameter count: 31
MLP parameter count: 4097


## 9. Training loop with validation and early stopping

The loop tracks train/validation loss and ROC-AUC at each epoch. The model state with the lowest validation loss is restored when training finishes. This fixes a common issue in hand-written pipelines: calculating loss from a global target tensor rather than from the labels passed to the loss function.

In [14]:
def train_model(
    model: nn.Module,
    model_name: str,
    *,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_PATIENCE,
) -> tuple[nn.Module, pd.DataFrame, dict]:
    """Train a binary classifier, monitor validation loss, and restore the best checkpoint."""
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0,
    )

    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = []
    start_time = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        accumulated_loss = 0.0
        observations_seen = 0

        for batch_features, batch_labels in loader:
            batch_features = batch_features.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()

            batch_count = len(batch_features)
            accumulated_loss += loss.item() * batch_count
            observations_seen += batch_count

        train_loss = accumulated_loss / max(observations_seen, 1)
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_tensor.to(DEVICE))
            val_loss = criterion(val_logits, y_val_tensor.to(DEVICE)).item()

        train_probs = predict_probabilities(model, X_train_tensor)
        val_probs = predict_probabilities(model, X_val_tensor)
        train_auc = roc_auc_score(y_train_np.reshape(-1), train_probs)
        val_auc = roc_auc_score(y_val_np, val_probs)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_roc_auc": train_auc,
            "val_roc_auc": val_auc,
        })

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"{model_name:20s} | epoch {epoch:03d} | "
                f"train loss {train_loss:.4f} | val loss {val_loss:.4f} | val ROC-AUC {val_auc:.4f}"
            )

        if epochs_without_improvement >= patience:
            print(f"Early stopping: no validation-loss improvement for {patience} epochs.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    elapsed = time.perf_counter() - start_time
    result_info = {
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "epochs_trained": len(history),
        "training_seconds": elapsed,
    }
    return model, pd.DataFrame(history), result_info

## 10. Train both models

Both candidates use the same training rows and early-stopping policy. A separate model instance is created for each candidate. The best epoch is selected by validation loss, and the candidates are compared using validation metrics.

In [15]:
set_seed(SEED)
logistic_model, logistic_history, logistic_info = train_model(
    build_model("Logistic Regression", X_train_tensor.shape[1]), "Logistic Regression"
)

set_seed(SEED)
mlp_model, mlp_history, mlp_info = train_model(
    build_model("MLP", X_train_tensor.shape[1]), "MLP"
)

models = {
    "Logistic Regression": logistic_model,
    "MLP": mlp_model,
}
histories = {
    "Logistic Regression": logistic_history,
    "MLP": mlp_history,
}
training_info = {
    "Logistic Regression": logistic_info,
    "MLP": mlp_info,
}

print("\nTraining completed.")

Logistic Regression  | epoch 001 | train loss 0.6351 | val loss 0.6071 | val ROC-AUC 0.7552
Logistic Regression  | epoch 010 | train loss 0.3203 | val loss 0.2976 | val ROC-AUC 0.9936
Logistic Regression  | epoch 020 | train loss 0.2179 | val loss 0.2013 | val ROC-AUC 0.9977
Logistic Regression  | epoch 030 | train loss 0.1717 | val loss 0.1595 | val ROC-AUC 0.9988
Logistic Regression  | epoch 040 | train loss 0.1459 | val loss 0.1345 | val ROC-AUC 0.9994
Logistic Regression  | epoch 050 | train loss 0.1298 | val loss 0.1187 | val ROC-AUC 0.9994
Logistic Regression  | epoch 060 | train loss 0.1188 | val loss 0.1080 | val ROC-AUC 0.9994
Logistic Regression  | epoch 070 | train loss 0.1106 | val loss 0.0997 | val ROC-AUC 0.9994
Logistic Regression  | epoch 080 | train loss 0.1045 | val loss 0.0937 | val ROC-AUC 0.9994
Logistic Regression  | epoch 090 | train loss 0.0994 | val loss 0.0889 | val ROC-AUC 0.9988
Logistic Regression  | epoch 100 | train loss 0.0954 | val loss 0.0852 | val ROC

In [16]:
# Interactive training diagnostics: loss on the left, ROC-AUC on the right.
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Loss by Epoch", "ROC-AUC by Epoch"),
    horizontal_spacing=0.12,
)
line_styles = {"Logistic Regression": "solid", "MLP": "dash"}
for model_name, hist in histories.items():
    fig.add_trace(go.Scatter(
        x=hist["epoch"], y=hist["train_loss"], mode="lines",
        name=f"{model_name} · train loss", line={"dash": line_styles[model_name]},
        legendgroup=model_name,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=hist["epoch"], y=hist["val_loss"], mode="lines",
        name=f"{model_name} · val loss", line={"dash": "dot"},
        legendgroup=model_name,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=hist["epoch"], y=hist["train_roc_auc"], mode="lines",
        name=f"{model_name} · train AUC", line={"dash": line_styles[model_name]},
        legendgroup=model_name,
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=hist["epoch"], y=hist["val_roc_auc"], mode="lines",
        name=f"{model_name} · val AUC", line={"dash": "dot"},
        legendgroup=model_name,
    ), row=1, col=2)
fig.update_xaxes(title_text="Epoch", rangemode="tozero")
fig.update_yaxes(title_text="Binary cross-entropy loss", row=1, col=1)
fig.update_yaxes(title_text="ROC-AUC", range=[0.45, 1.02], row=1, col=2)
fig.update_layout(title="Training Diagnostics — Learning and Generalization", height=500, title_x=0.02)
publish_plot(fig, "07_training_diagnostics")

Interactive chart saved: outputs/plots/07_training_diagnostics.html


PosixPath('outputs/plots/07_training_diagnostics.html')

## 11. Compare candidates on validation data

Validation ROC-AUC summarizes ranking quality across probability thresholds. Validation average precision is included as a second ranking metric. F1 at 0.50 is included as a threshold-specific reference—not as a universal decision rule.

In [17]:
validation_rows = []
for model_name, model in models.items():
    val_probabilities = predict_probabilities(model, X_val_tensor)
    val_predictions_050 = (val_probabilities >= 0.50).astype(int)
    info = training_info[model_name]
    validation_rows.append({
        "Model": model_name,
        "Validation ROC-AUC": roc_auc_score(y_val_np, val_probabilities),
        "Validation average precision": average_precision_score(y_val_np, val_probabilities),
        "Validation F1 @ 0.50": f1_score(y_val_np, val_predictions_050, zero_division=0),
        "Best epoch": info["best_epoch"],
        "Epochs trained": info["epochs_trained"],
        "Training seconds": round(info["training_seconds"], 2),
    })

validation_results = pd.DataFrame(validation_rows).sort_values("Validation ROC-AUC", ascending=False).reset_index(drop=True)
display(validation_results.style.format({
    "Validation ROC-AUC": "{:.4f}",
    "Validation average precision": "{:.4f}",
    "Validation F1 @ 0.50": "{:.4f}",
    "Training seconds": "{:.2f}",
}))

long_validation = validation_results.melt(
    id_vars="Model",
    value_vars=["Validation ROC-AUC", "Validation average precision", "Validation F1 @ 0.50"],
    var_name="Metric", value_name="Score",
)
fig = px.bar(
    long_validation, x="Model", y="Score", color="Metric", barmode="group",
    text=long_validation["Score"].map(lambda value: f"{value:.3f}"),
    title="Candidate Model Comparison — Validation Metrics",
    range_y=[0, 1.08],
)
fig.update_traces(textposition="outside")
fig.update_layout(title_x=0.02)
publish_plot(fig, "08_validation_model_comparison")

,Model,Validation ROC-AUC,Validation average precision,Validation F1 @ 0.50,Best epoch,Epochs trained,Training seconds
0,MLP,1.0000,1.0000,0.9846,77,95,3.35
1,Logistic Regression,0.9983,0.9973,0.9841,120,120,3.38


Interactive chart saved: outputs/plots/08_validation_model_comparison.html


PosixPath('outputs/plots/08_validation_model_comparison.html')

In [18]:
# Select by validation ROC-AUC only. The test split has not been used for model selection.
best_model_name = validation_results.iloc[0]["Model"]
best_model = models[best_model_name]
val_probabilities = predict_probabilities(best_model, X_val_tensor)
print(f"Selected model: {best_model_name}")
print(f"Validation ROC-AUC: {roc_auc_score(y_val_np, val_probabilities):.4f}")

Selected model: MLP
Validation ROC-AUC: 1.0000


## 12. Choose a demonstration threshold on validation data

A probability threshold converts a score into a class label. Instead of hard-coding an arbitrary value such as `0.90`, this notebook inspects a threshold range and selects the threshold with the highest **validation F1-score** (breaking ties in favor of higher sensitivity/recall). This is a teaching example, not a clinically suitable threshold-selection strategy; real deployments require domain review and a defined cost of errors.

In [19]:
threshold_rows = []
threshold_grid = np.round(np.arange(0.10, 0.901, 0.01), 2)
for threshold in threshold_grid:
    predictions = (val_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val_np, predictions, labels=[0, 1]).ravel()
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val_np, predictions, zero_division=0),
        "Sensitivity": recall_score(y_val_np, predictions, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "F1-score": f1_score(y_val_np, predictions, zero_division=0),
    })
threshold_results = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_results.sort_values(
    ["F1-score", "Sensitivity"], ascending=[False, False]
).iloc[0]
DECISION_THRESHOLD = float(best_threshold_row["Threshold"])
print(f"Selected validation threshold: {DECISION_THRESHOLD:.2f}")
print(f"Validation F1 at selected threshold: {best_threshold_row['F1-score']:.4f}")
print("Threshold will be held fixed for the final test evaluation.")

Selected validation threshold: 0.56
Validation F1 at selected threshold: 1.0000
Threshold will be held fixed for the final test evaluation.


In [20]:
fig = go.Figure()
for metric in ["Precision", "Sensitivity", "Specificity", "F1-score"]:
    fig.add_trace(go.Scatter(
        x=threshold_results["Threshold"], y=threshold_results[metric],
        mode="lines", name=metric,
    ))
fig.add_vline(
    x=DECISION_THRESHOLD, line_dash="dash", line_width=2,
    annotation_text=f"Selected = {DECISION_THRESHOLD:.2f}", annotation_position="top left",
)
fig.update_xaxes(title="Probability threshold", range=[0.1, 0.9])
fig.update_yaxes(title="Validation score", range=[0, 1.02])
fig.update_layout(title="Validation Threshold Analysis", title_x=0.02, height=480)
publish_plot(fig, "09_validation_threshold_analysis")

Interactive chart saved: outputs/plots/09_validation_threshold_analysis.html


PosixPath('outputs/plots/09_validation_threshold_analysis.html')

## 13. Final evaluation on the held-out test set

The selected model and validation-selected threshold are now fixed. The test set is evaluated once for the final report. The **positive class is malignant**, so sensitivity is the fraction of malignant benchmark examples predicted as malignant, and specificity is the fraction of benign examples predicted as benign.

In [21]:
test_probabilities = predict_probabilities(best_model, X_test_tensor)
test_predictions = (test_probabilities >= DECISION_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_np, test_predictions, labels=[0, 1]).ravel()

test_metrics = {
    "Selected model": best_model_name,
    "Decision threshold": DECISION_THRESHOLD,
    "Accuracy": accuracy_score(y_test_np, test_predictions),
    "Precision": precision_score(y_test_np, test_predictions, zero_division=0),
    "Sensitivity (malignant recall)": recall_score(y_test_np, test_predictions, zero_division=0),
    "Specificity (benign recall)": tn / (tn + fp) if (tn + fp) else 0.0,
    "F1-score": f1_score(y_test_np, test_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test_np, test_probabilities),
    "Average precision": average_precision_score(y_test_np, test_probabilities),
    "True benign (TN)": int(tn),
    "False malignant (FP)": int(fp),
    "Missed malignant (FN)": int(fn),
    "True malignant (TP)": int(tp),
}

test_metrics_df = pd.DataFrame({"Metric": list(test_metrics.keys()), "Value": list(test_metrics.values())})
display(test_metrics_df.style.format({"Value": lambda value: f"{value:.4f}" if isinstance(value, (float, np.floating)) else str(value)}))

,Metric,Value
0,Selected model,MLP
1,Decision threshold,0.5600
2,Accuracy,1.0000
3,Precision,1.0000
4,Sensitivity (malignant recall),1.0000
5,Specificity (benign recall),1.0000
6,F1-score,1.0000
7,ROC-AUC,1.0000
8,Average precision,1.0000
9,True benign (TN),54


In [22]:
# ROC curve on the held-out test data.
fpr, tpr, roc_thresholds = roc_curve(y_test_np, test_probabilities)
test_auc = roc_auc_score(y_test_np, test_probabilities)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines",
    name=f"Selected model (AUC = {test_auc:.3f})",
    line={"width": 3},
    hovertemplate="False positive rate: %{x:.3f}<br>True positive rate: %{y:.3f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random-ranking reference",
    line={"dash": "dash", "color": "gray"},
))
fig.update_xaxes(title="False positive rate (1 − specificity)", range=[0, 1])
fig.update_yaxes(title="True positive rate (sensitivity)", range=[0, 1.02])
fig.update_layout(title="Test ROC Curve", title_x=0.02, height=520)
publish_plot(fig, "10_test_roc_curve")

Interactive chart saved: outputs/plots/10_test_roc_curve.html


PosixPath('outputs/plots/10_test_roc_curve.html')

In [23]:
# Precision-recall curve on the held-out test data.
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test_np, test_probabilities)
average_precision = average_precision_score(y_test_np, test_probabilities)
positive_rate = float(y_test_np.mean())
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall_curve, y=precision_curve, mode="lines",
    name=f"Selected model (AP = {average_precision:.3f})",
    line={"width": 3},
    hovertemplate="Recall: %{x:.3f}<br>Precision: %{y:.3f}<extra></extra>",
))
fig.add_hline(
    y=positive_rate, line_dash="dash", line_color="gray",
    annotation_text=f"Positive prevalence = {positive_rate:.2f}", annotation_position="bottom right",
)
fig.update_xaxes(title="Recall / sensitivity", range=[0, 1])
fig.update_yaxes(title="Precision", range=[0, 1.02])
fig.update_layout(title="Test Precision–Recall Curve", title_x=0.02, height=520)
publish_plot(fig, "11_test_precision_recall_curve")

Interactive chart saved: outputs/plots/11_test_precision_recall_curve.html


PosixPath('outputs/plots/11_test_precision_recall_curve.html')

In [24]:
# Confusion matrix: rows are actual classes, columns are predicted classes.
cm = confusion_matrix(y_test_np, test_predictions, labels=[0, 1])
class_names = ["Benign", "Malignant"]
fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=[f"Predicted {name}" for name in class_names],
    y=[f"Actual {name}" for name in class_names],
    text=cm,
    texttemplate="%{text}",
    textfont={"size": 18},
    colorscale="Blues",
    hovertemplate="%{y}<br>%{x}<br>Count: %{z}<extra></extra>",
    showscale=False,
))
fig.update_layout(title="Test Confusion Matrix", title_x=0.02, height=470)
publish_plot(fig, "12_test_confusion_matrix")

Interactive chart saved: outputs/plots/12_test_confusion_matrix.html


PosixPath('outputs/plots/12_test_confusion_matrix.html')

In [25]:
# Look at the model's probability scores, separated by the true class.
probability_df = pd.DataFrame({
    "Predicted probability of malignant": test_probabilities,
    "Actual class": pd.Series(y_test_np).map({0: "Benign", 1: "Malignant"}),
    "Predicted class": pd.Series(test_predictions).map({0: "Benign", 1: "Malignant"}),
})
fig = px.histogram(
    probability_df,
    x="Predicted probability of malignant",
    color="Actual class",
    marginal="rug",
    barmode="overlay",
    opacity=0.62,
    nbins=24,
    title="Test Probability Scores by Actual Class",
    labels={"Actual class": "Actual diagnosis"},
)
fig.add_vline(x=DECISION_THRESHOLD, line_dash="dash", line_width=2,
              annotation_text=f"Decision threshold = {DECISION_THRESHOLD:.2f}")
fig.update_layout(title_x=0.02)
publish_plot(fig, "13_test_probability_distribution")

Interactive chart saved: outputs/plots/13_test_probability_distribution.html


PosixPath('outputs/plots/13_test_probability_distribution.html')

## 14. Feature importance with permutation

Neural networks do not have a simple coefficient table like a linear model. Permutation importance estimates how much validation ROC-AUC drops when one feature is randomly shuffled while the others stay unchanged. A larger drop suggests the model relied more on that feature in this validation sample. Correlated features can share or mask importance, and this ranking is not causal evidence.

In [26]:
rng = np.random.default_rng(SEED)
base_validation_auc = roc_auc_score(y_val_np, val_probabilities)
PERMUTATION_REPEATS = 6
importance_rows = []

for feature_index, feature_name in enumerate(FEATURE_COLUMNS):
    auc_drops = []
    for _ in range(PERMUTATION_REPEATS):
        shuffled_features = X_val_scaled.copy()
        shuffled_features[:, feature_index] = rng.permutation(shuffled_features[:, feature_index])
        shuffled_tensor = torch.tensor(shuffled_features, dtype=torch.float32)
        shuffled_probabilities = predict_probabilities(best_model, shuffled_tensor)
        shuffled_auc = roc_auc_score(y_val_np, shuffled_probabilities)
        auc_drops.append(base_validation_auc - shuffled_auc)
    importance_rows.append({
        "Feature": feature_name,
        "Mean AUC drop": float(np.mean(auc_drops)),
        "Std AUC drop": float(np.std(auc_drops, ddof=1)) if len(auc_drops) > 1 else 0.0,
    })

importance_df = pd.DataFrame(importance_rows).sort_values("Mean AUC drop", ascending=False).reset_index(drop=True)
top_importance = importance_df.head(15).sort_values("Mean AUC drop", ascending=True)
fig = go.Figure(go.Bar(
    x=top_importance["Mean AUC drop"],
    y=top_importance["Feature"],
    orientation="h",
    error_x={"type": "data", "array": top_importance["Std AUC drop"], "visible": True},
    hovertemplate="%{y}<br>Mean AUC drop: %{x:.4f}<extra></extra>",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_xaxes(title="Decrease in validation ROC-AUC after shuffling")
fig.update_yaxes(title="Feature")
fig.update_layout(title="Top 15 Features — Permutation Importance on Validation Data", height=650, title_x=0.02)
publish_plot(fig, "14_validation_permutation_importance")
display(importance_df.head(15).style.format({"Mean AUC drop": "{:.4f}", "Std AUC drop": "{:.4f}"}))

Interactive chart saved: outputs/plots/14_validation_permutation_importance.html


,Feature,Mean AUC drop,Std AUC drop
0,texture_worst,0.0051,0.0044
1,compactness_se,0.0038,0.0056
2,symmetry_worst,0.0033,0.0028
3,concavity_worst,0.0027,0.0019
4,concave points_worst,0.0017,0.0021
5,area_worst,0.0016,0.0013
6,smoothness_worst,0.0016,0.0009
7,radius_se,0.0012,0.0014
8,area_se,0.0012,0.0011
9,concavity_mean,0.0012,0.0008


## 15. Demonstration inference

The next cell runs inference for one held-out example and reports the model's probability output, thresholded prediction, and known benchmark label. It is only a software demonstration; do not interpret the probability as a calibrated clinical risk estimate.

In [27]:
sample_raw = X_test_raw.iloc[[0]].copy()
sample_imputed = imputer.transform(sample_raw)
sample_scaled = scaler.transform(sample_imputed).astype(np.float32)
sample_tensor = torch.tensor(sample_scaled, dtype=torch.float32)
sample_probability = float(predict_probabilities(best_model, sample_tensor)[0])
sample_prediction = int(sample_probability >= DECISION_THRESHOLD)
sample_actual = int(y_test.iloc[0])

inference_demo = pd.DataFrame([{
    "Example": "Held-out benchmark row 1",
    "Predicted probability of malignant": sample_probability,
    "Decision threshold": DECISION_THRESHOLD,
    "Predicted benchmark class": {0: "Benign", 1: "Malignant"}[sample_prediction],
    "Actual benchmark class": {0: "Benign", 1: "Malignant"}[sample_actual],
    "Prediction matches label": sample_prediction == sample_actual,
}])
display(inference_demo.style.format({
    "Predicted probability of malignant": "{:.4f}",
    "Decision threshold": "{:.2f}",
}))

,Example,Predicted probability of malignant,Decision threshold,Predicted benchmark class,Actual benchmark class,Prediction matches label
0,Held-out benchmark row 1,0.0001,0.56,Benign,Benign,True


## 16. Save the trained model and preprocessing artifacts

The trained weights alone are not enough to reproduce predictions. The scaler, imputer, ordered feature names, target convention, model class, and selected threshold are saved too. Keep these files together and use trusted sources when loading serialized artifacts.

In [28]:
MODEL_PATH = ARTIFACT_DIR / "best_pytorch_breast_cancer_classifier.pt"
PREPROCESSING_PATH = ARTIFACT_DIR / "preprocessing.joblib"
METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"

checkpoint = {
    "model_name": best_model_name,
    "input_dim": int(len(FEATURE_COLUMNS)),
    "model_state_dict": {key: value.detach().cpu().clone() for key, value in best_model.state_dict().items()},
    "decision_threshold": float(DECISION_THRESHOLD),
    "target_positive_class": "Malignant (M=1)",
    "target_negative_class": "Benign (B=0)",
    "seed": int(SEED),
}
torch.save(checkpoint, MODEL_PATH)
joblib.dump({
    "imputer": imputer,
    "scaler": scaler,
    "feature_columns": FEATURE_COLUMNS,
}, PREPROCESSING_PATH)

metadata = {
    "project": "PyTorch Breast Cancer Classification Learning Pipeline",
    "data_source": DATA_SOURCE,
    "model_name": best_model_name,
    "feature_count": len(FEATURE_COLUMNS),
    "feature_columns": FEATURE_COLUMNS,
    "target_mapping": {"0": "Benign (B)", "1": "Malignant (M)"},
    "decision_threshold_selected_on_validation": DECISION_THRESHOLD,
    "validation_roc_auc": float(roc_auc_score(y_val_np, val_probabilities)),
    "test_metrics": {key: (float(value) if isinstance(value, (np.floating, float)) else int(value) if isinstance(value, (np.integer, int)) else value)
                     for key, value in test_metrics.items()},
    "educational_use_only": True,
    "clinical_use_validated": False,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved artifacts:")
for path in [MODEL_PATH, PREPROCESSING_PATH, METADATA_PATH]:
    print(f" - {path.resolve()} ({path.stat().st_size:,} bytes)")
print(f" - Interactive plots: {len(list(PLOT_DIR.glob('*.html')))} HTML files in {PLOT_DIR.resolve()}")

Saved artifacts:
 - /content/artifacts/best_pytorch_breast_cancer_classifier.pt (19,965 bytes)
 - /content/artifacts/preprocessing.joblib (3,146 bytes)
 - /content/artifacts/model_metadata.json (1,551 bytes)
 - Interactive plots: 14 HTML files in /content/outputs/plots


In [29]:
# Optional round-trip verification: reload the saved weights and preprocessing objects, then reproduce the demo score.
try:
    reloaded_checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
except TypeError:  # Compatibility with older PyTorch versions
    reloaded_checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

reloaded_model = build_model(reloaded_checkpoint["model_name"], reloaded_checkpoint["input_dim"])
reloaded_model.load_state_dict(reloaded_checkpoint["model_state_dict"])
reloaded_model = reloaded_model.to(DEVICE).eval()
reloaded_preprocessing = joblib.load(PREPROCESSING_PATH)

reloaded_raw = X_test_raw.iloc[[0]].reindex(columns=reloaded_preprocessing["feature_columns"])
reloaded_scaled = reloaded_preprocessing["scaler"].transform(
    reloaded_preprocessing["imputer"].transform(reloaded_raw)
).astype(np.float32)
reloaded_probability = float(predict_probabilities(reloaded_model, torch.tensor(reloaded_scaled, dtype=torch.float32))[0])
print(f"Original in-memory score: {sample_probability:.8f}")
print(f"Reloaded artifact score:  {reloaded_probability:.8f}")
print(f"Round-trip absolute difference: {abs(sample_probability - reloaded_probability):.10f}")
assert np.isclose(sample_probability, reloaded_probability, atol=1e-6), "Saved model round-trip check failed."
print("Artifact round-trip check passed.")

Original in-memory score: 0.00008070
Reloaded artifact score:  0.00008070
Round-trip absolute difference: 0.0000000000
Artifact round-trip check passed.


## 17. Interpretation and next steps

### What to take away

- A strong machine-learning workflow separates training, validation, and test data before fitting preprocessing steps.
- `BCEWithLogitsLoss` should be paired with model outputs that are raw logits; apply sigmoid when probabilities are needed for metrics or inference.
- Validation data is appropriate for early stopping, model selection, and demonstration threshold tuning. The held-out test set is reserved for the final report.
- Accuracy alone can hide class-specific errors. Review sensitivity, specificity, precision, F1, ROC-AUC, average precision, and the confusion matrix together.
- Interactive Plotly charts are saved in `outputs/plots/` as HTML files. Open an HTML file in a browser; the Plotly JavaScript library is loaded from a CDN, so an internet connection may be needed to render the saved chart.
- Permutation importance reports a model-performance association on this split; it is not causal evidence, and correlated measurements can make rankings unstable.

### Limitations

This is a small, historical benchmark dataset. Its sample size, collection context, feature definitions, and representativeness limit generalization. Results may change with a different seed, split, data source, or library version. There is no external validation, calibration study, prospective evaluation, fairness analysis, or clinical governance in this notebook.

### Ideas for further practice

1. Add stratified cross-validation and report mean ± standard deviation for each metric.
2. Compare against scikit-learn's logistic regression and a tree-based model as additional baselines.
3. Add probability calibration and reliability diagrams.
4. Log experiments (seed, architecture, learning rate, batch size, threshold policy) to a CSV file.
5. Add unit tests for data columns, target mapping, artifact reload, and inference input validation.

---

**End of notebook.** Re-run from the top for a reproducible experiment. Model scores are benchmark results for learning, not clinical performance claims.